In [ ]:
import os

data_dir = os.path.abspath("data/public")
project_dir = os.path.abspath("data/project")
chrombpnet_dir = os.path.join(project_dir, "chrombpnet")

In [ ]:
import torch
from bpnetlite.chrombpnet import ChromBPNet
from bpnetlite.bpnet import BPNet
import tangermeme
import numpy as np
import pandas as pd
import h5py
import torch.nn as nn
import scipy.io
import os
from tqdm.notebook import tqdm
from statsmodels.stats.multitest import multipletests
import pickle
from scipy.stats import hypergeom

In [ ]:
# Modified from here: https://github.com/jmschrei/bpnet-lite/blob/master/bpnetlite/bpnet.py#L551
def import_chrombpnet_accessibility(filename, model_name = 'wo_bias'):
    with h5py.File(filename, "r") as h5:
        w = h5['model_weights']
        if model_name == 'wo_bias':
            prefix = 'wo_bias_'
            model_name = 'model_wo_bias'
        else:
            prefix = ''
            model_name = 'model'
        namer = lambda model_name, prefix, suffix: '{0}/{1}{2}'.format(model_name, prefix, suffix)
        k, b = 'kernel:0', 'bias:0'
        n_layers = 0
        for layer_name in w[model_name].keys():
            try:
                idx = int(layer_name.split("_")[-1].replace("conv", ""))
                n_layers = max(n_layers, idx)
            except:
                pass
        name = namer(model_name, prefix, "bpnet_1conv")
        n_filters = w[name][k].shape[2]

        model = BPNet(n_layers=n_layers, n_filters=n_filters, n_outputs=1, n_control_tracks=0)
        convert_w = lambda x: torch.nn.Parameter(torch.tensor(x[:]).permute(2, 1, 0))
        convert_b = lambda x: torch.nn.Parameter(torch.tensor(x[:]))

        iname = namer(model_name, prefix, 'bpnet_1st_conv')

        model.iconv.weight = convert_w(w[iname][k])
        model.iconv.bias = convert_b(w[iname][b])
        model.iconv.padding = ((21 - 1) // 2,)

        for i in range(1, n_layers+1):
            lname = namer(model_name, prefix, 'bpnet_{}conv'.format(i))
            model.rconvs[i-1].weight = convert_w(w[lname][k])
            model.rconvs[i-1].bias = convert_b(w[lname][b])

        if model_name == 'model_wo_bias':
            prefix = 'wo_bias_bpnet_'
        fname = namer(model_name, prefix, 'prof_out_precrop')
        model.fconv.weight = convert_w(w[fname][k])
        model.fconv.bias = convert_b(w[fname][b])
        model.fconv.padding = ((75 - 1) // 2,)
        name = namer(model_name, prefix, "logcount_predictions")
        model.linear.weight = torch.nn.Parameter(torch.tensor(w[name][k][:].T))
        model.linear.bias = convert_b(w[name][b])
        return model

In [ ]:
# Modified from here: https://github.com/jmschrei/bpnet-lite/blob/master/bpnetlite/bpnet.py#L551
import re
def import_chrombpnet_bias(filename):
    with h5py.File(filename, "r") as h5:
        w = h5['model_weights']
        namer = lambda prefix, suffix: '{0}{1}/{0}{1}'.format(prefix, suffix)
        prefix = ""
        k, b = 'kernel:0', 'bias:0'
        nums = []
        for layer_name in w.keys():
            try:
                match = re.match(r"bpnet_(\d+)conv", layer_name)
                if match:
                    nums.append(int(match.group(1)))
            except:
                pass
        n_layers = max(nums)
        name = namer(prefix, "bpnet_1conv")
        n_filters = w[name][k].shape[2]

        model = BPNet(n_layers=n_layers, n_filters=n_filters, n_outputs=1, n_control_tracks=0)
        convert_w = lambda x: torch.nn.Parameter(torch.tensor(x[:]).permute(2, 1, 0))
        convert_b = lambda x: torch.nn.Parameter(torch.tensor(x[:]))

        iname = namer(prefix, 'bpnet_1st_conv')

        model.iconv.weight = convert_w(w[iname][k])
        model.iconv.bias = convert_b(w[iname][b])
        model.iconv.padding = ((21 - 1) // 2,)

        for i in range(1, n_layers+1):
            lname = namer(prefix, 'bpnet_{}conv'.format(i))
            model.rconvs[i-1].weight = convert_w(w[lname][k])
            model.rconvs[i-1].bias = convert_b(w[lname][b])

        fname = namer(prefix, 'prof_out_precrop')
        model.fconv.weight = convert_w(w[fname][k])
        model.fconv.bias = convert_b(w[fname][b])
        model.fconv.padding = ((75 - 1) // 2,)
        name = namer(prefix, "logcount_predictions")
        model.linear.weight = torch.nn.Parameter(torch.tensor(w[name][k][:].T))
        model.linear.bias = convert_b(w[name][b])
        return model

In [ ]:
class ModelWrapper(nn.Module):
    def __init__(self, model, device=None):
        super().__init__()
        self.model = model
        self.device = device
        self.model.to(self.device)
        self.model.eval()

    def forward(self, x):
        x = x.float().to(self.device)
        with torch.no_grad(): 
            out = self.model(x)
        return out[1]

### Motifs

##### Find motifs

In [ ]:
from tangermeme.io import extract_loci
from tangermeme.deep_lift_shap import deep_lift_shap
from memelite.fimo import fimo, read_meme
window_size = 500
overwrite = False
import os
import pickle
genome_file = f"{chrombpnet_dir}/data/hg38.fa"
motif_path = f"{chrombpnet_dir}/data/motifs.meme"

# Loop over cell types and find peaks
celltypes = ['alpha', 'beta']
for celltype in celltypes:
    # Output file
    output = f"{chrombpnet_dir}/data/peaks/correlated/hits/{celltype}_all.motifs"
    if overwrite or not os.path.exists(output):       
        ## All peaks
        # Read bed file
        peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks/{celltype}_filtered.bed", sep="\t", header=None)
        peaks = peaks[[0,1,2]].copy()
        peaks = peaks.rename(columns={0: 'chromosome', 1: 'start', 2: 'end'})
    
        ## PC-associated peaks
        files = os.listdir(f"{chrombpnet_dir}/data/peaks/correlated/{celltype}/bed")
        for file in files:
            # Read bed file
            pc_peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks/correlated/{celltype}/bed/{file}", sep="\t", header=None)
            pc_peaks = pc_peaks[[0,1,2]].copy()
            pc_peaks = pc_peaks.rename(columns={0: 'chromosome', 1: 'start', 2: 'end'})
            pc_peaks['class'] = file.split(".")[0]
            peaks = pd.concat([peaks, pc_peaks])
    
        ## Collaps and track the source
        peaks = peaks.groupby(["chromosome", "start", "end"], as_index=False).agg({"class": lambda x: ",".join(x.dropna().astype(str))})
        peaks["class"] = peaks["class"].replace("", "baseline")
        peaks.to_csv(f"{chrombpnet_dir}/data/peaks/correlated/hits/{celltype}_peaks.csv", header=True)
    
        # Extract data
        X = extract_loci(peaks[['chromosome','start','end']], genome_file, ignore=['N','B'])
    
        # Find hits
        hits = fimo(motif_path, X[:,:,(1057 - (window_size//2)):(1057 + (window_size//2))])
    
        # Output
        with open(output, "wb") as f:
            pickle.dump(hits, f)

##### Calculate enrichment

In [ ]:
### Enrichment analyes
import pickle
from memelite.fimo import read_meme
from tqdm.notebook import tqdm
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

# Setup
per_pc = False
celltypes = ['beta', 'alpha']
motif_path = f"{chrombpnet_dir}/data/motifs.meme"
background = 'baseline'
sample_background = True
overwrite = False
output = f"{chrombpnet_dir}/data/peaks/correlated/hits/motif_enrichments.pkl"

# Load motifs
motifs = read_meme(motif_path)

if overwrite or not os.path.exists(output):
    # Define results dictionary
    res_dict = {}
    
    # Calculate enrichments
    for celltype in celltypes:
        # Read bed file
        peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks/correlated/hits/{celltype}_peaks.csv", index_col= 0)
    
        # Read motif hits
        with open(f"{chrombpnet_dir}/data/peaks/correlated/hits/{celltype}_all.motifs", "rb") as f:
            hits = pickle.load(f)
    
        # Define foreground groups
        foreground_values = list(set(peaks["class"].dropna().str.split(",").explode()) - set(['baseline']))
        if per_pc:
            foreground_values = list(set([x.split("_")[0] + "_" for x in foreground_values]))
    
        # Define foreground and backgrounds
        test_sets = {}
        for foreground_value in foreground_values:
            # Define foreground
            foreground_ids = peaks[peaks['class'].str.contains(rf'\b{foreground_value}\S*\b', na=False)].index.tolist()
    
            # Define background
            if background == 'baseline':
                background_peaks = peaks[peaks['class'] == "baseline"]
            elif background == 'others':
                background_peaks = peaks[~peaks.index.isin(foreground_ids)]
                background_peaks = background_peaks[background_peaks['class'] != "baseline"]
            else:
                background_peaks = peaks[~peaks.index.isin(foreground_ids)]
    
            # Sample if required
            if sample_background:
                background_ids = background_peaks.sample(n=len(foreground_ids), random_state=42).index.tolist()
            else:
                background_ids = background_peaks.index.tolist()
    
            # Store in dictionary
            test_sets[foreground_value] = {
                'foreground': foreground_ids,
                'background': background_ids
            }
    
        # Setup results dictionary
        res_dict[celltype] = {}
        for foreground_value in foreground_values:
            res_dict[celltype][foreground_value] = pd.DataFrame(columns=['Motif','Pvalue','Foreground','Background','nForeground'])
    
        # Test all enrichments
        counter = 0
        for idx in tqdm(range(len(hits)), desc="Processing motifs"):
            # Setup
            motif_name = list(motifs.keys())[idx].split(" ")[0]
            targets = hits[idx].copy()
        
            # Loop over foregrounds
            for foreground_value in foreground_values:
                # Calculate stats
                n = len(test_sets[foreground_value]['foreground'])
                k = len(set(targets['sequence_name'].tolist()).intersection(test_sets[foreground_value]['foreground']))
                N = len(test_sets[foreground_value]['background']) + n 
                K = len(set(targets['sequence_name'].tolist()).intersection(test_sets[foreground_value]['background'])) + k
                pval = hypergeom.sf(k-1, N, K, n)
                jacc_foreground = len(set(targets['sequence_name'].tolist()).intersection(test_sets[foreground_value]['foreground'])) / len(set(targets['sequence_name'].tolist()).union(test_sets[foreground_value]['foreground']))
                jacc_background = len(set(targets['sequence_name'].tolist()).intersection(test_sets[foreground_value]['background'])) / len(set(targets['sequence_name'].tolist()).union(test_sets[foreground_value]['background']))
            
                # Setup results
                tmp_df = pd.DataFrame({
                    'Motif': motif_name,
                    'Set': foreground_value,
                    'Pvalue': pval,
                    'Foreground': jacc_foreground,
                    'Background': jacc_background,
                    'nForeground': k
                }, index = [counter])
                res_dict[celltype][foreground_value] = pd.concat([res_dict[celltype][foreground_value], tmp_df])
                counter += 1

        # Save to pkl
        with open(output, "wb") as f:
            pickle.dump(res_dict, f)
else:
    with open(output, "rb") as f:
        res_dict = pickle.load(f)

### Ablation analysis (using ChromBPNet)

#### Run ablations

##### Setup

In [ ]:
### Ablation analyses
from tangermeme.ablate import ablate_annotations
from tqdm.notebook import tqdm
from tangermeme.ersatz import shuffle
from tangermeme.predict import predict
from tangermeme.io import extract_loci
import itertools
import os
import pickle
from memelite.fimo import read_meme, fimo
window_size = 500
import copy

# Setup
batch_size = 256
celltypes = ['beta', 'alpha']
window_size = 500
folds = [0,1,2,3,4]
states = ['ND', 'T2D']
total = len(states) * len(folds)
n_resamples = 5
genome_file = f"{chrombpnet_dir}/data/hg38.fa"
motif_path = f"{chrombpnet_dir}/data/motifs.meme"
output_path = f"{chrombpnet_dir}/data/peaks/correlated/"

# Load motifs
motifs = read_meme(motif_path)

##### Process peaks

In [ ]:
# Combine peaks
peaks_df = []
for celltype in celltypes:
    # Read bed file
    peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks/correlated/hits/{celltype}_peaks.csv", index_col= 0)
    peaks = peaks[peaks['class'] != "baseline"]
    peaks["class"] = peaks["class"].str.split(",").apply(lambda xs: [f"{celltype}_{x}" for x in xs]).str.join(",")
    peaks_df.append(peaks)

all_peaks = pd.concat(peaks_df, ignore_index=True)
all_peaks = all_peaks.groupby(["chromosome", "start", "end"], as_index=False).agg({"class": lambda x: ",".join(x.astype(str))})
all_peaks = all_peaks.reset_index(drop=True)

In [ ]:
# Extract data
X = extract_loci(all_peaks[['chromosome','start','end']], genome_file, ignore=['N','B'])

# Subset
all_peaks = all_peaks.iloc[np.argwhere(X.sum(axis = 1).min(axis = 1).values > 0)[0,:]]
X = X[np.argwhere(X.sum(axis = 1).min(axis = 1).values > 0)[0,:],:,:]

# Find motif hits
hits = fimo(motif_path, X[:,:,(1057 - (window_size//2)):(1057 + (window_size//2))])

# Process peaks into sets based on annotation
peaksets = {}
for celltype in celltypes:
    peaksets[celltype] = set(all_peaks[all_peaks['class'].str.contains(rf'\b{celltype}\S*\b', na=False)].index.tolist())

common = set.intersection(*peaksets.values())
for k in peaksets:
    peaksets[k] -= common
    peaksets[k] = list(peaksets[k])
    
peaksets['common'] = list(common)

##### Create dict of models

In [ ]:
model_dict = {}
for celltype in celltypes:
    model_dict[celltype] = {}
    for state in states:
        model_dict[celltype][state] = {}
        for fold in folds:
            model_dict[celltype][state][fold] = {}
    
for celltype, state, fold in itertools.product(celltypes, states, folds):
    # Load and wrap model
    bias = import_chrombpnet_bias(f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_{state}_bias_model_scaled.h5")
    acc = import_chrombpnet_accessibility(f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}_{state}.h5", model_name = 'wo_bias')
    model = ChromBPNet(bias, acc, f"{celltype}_{fold}")
    model_dict[celltype][state][fold] = ModelWrapper(model, 'cuda')

##### Create dict for results

In [ ]:
# Create results dictionary
result_dict = {}
for celltype in celltypes:
    result_dict[celltype] = {}
    for state in states:
        result_dict[celltype][state] = {}
        for fold in folds:
            result_dict[celltype][state][fold] = { 'before': np.array([]), 'after': np.array([]), 'idx': np.array([]) }

# Create a copy
res_dict = copy.deepcopy(result_dict)
temp_dict = copy.deepcopy(result_dict)

##### Run ablations

In [ ]:
overwrite = False
for midx in tqdm(range(len(hits)), desc="Processing motifs"):
    output = f"{output_path}/old/{midx}.pkl"
    if overwrite or not os.path.exists(output):
        # Setup
        result_dict = copy.deepcopy(res_dict)
        motif_name = list(motifs.keys())[midx].split(" ")[0]
        targets = hits[midx].copy()
    
        # Process each peakset
        for k in peaksets:
            # Define models to use
            models = [k]
            if models[0] == 'common':
                models = ['alpha', 'beta']
    
            # Define the targets
            peakset_ids = peaksets[k]
            peakset_targets = targets[targets['sequence_name'].isin(peakset_ids)].copy()
    
            # Extract and perturb sequences
            if len(peakset_targets) > 0:
                peakset_targets['start'] = peakset_targets['start'] + (1057 - (window_size//2))
                peakset_targets['end'] = peakset_targets['end'] + (1057 - (window_size//2))
                annotation_tensor = torch.tensor(np.array(peakset_targets[['sequence_name', 'start', 'end']]))
        
                # Loop through annotation and ablate
                X_perturb = torch.empty(len(annotation_tensor), n_resamples, 4, 2114)
                X_original = torch.empty(len(annotation_tensor), 4, 2114)
                for i, (idx, start, end) in enumerate(annotation_tensor):
                    # Perturb the sequence
                    X_perturb[i,:,:,:] = shuffle(X[idx:idx+1], start=start, end=end, n=n_resamples, random_state=None)
                    X_original[i,:,:] = X[idx:idx+1]
    
                # Preallocate numpy arrays
                for celltype, state, fold in itertools.product(models, states, folds):
                    temp_dict[celltype][state][fold]['idx'] = np.empty((len(X_original)))
                    temp_dict[celltype][state][fold]['before'] = np.empty((len(X_original)))
                    temp_dict[celltype][state][fold]['after'] = np.empty((len(X_original), n_resamples))
                
                # Predict for original across all models
                for i in range(0, len(X_original), batch_size):
                    batch = X_original[i : (i+batch_size),:,:].to('cuda')
                    for celltype, state, fold in itertools.product(models, states, folds):
                        prediction = model_dict[celltype][state][fold](batch).cpu().numpy()[:,0]
                        temp_dict[celltype][state][fold]['before'][i:(i+len(batch))] = prediction
                        temp_dict[celltype][state][fold]['idx'][i:(i+len(batch))] = peakset_targets['sequence_name'].to_numpy()[i:(i+len(batch))]
            
                # Predict for perturbed across all models and resamples
                for samp in range(n_resamples):
                    for i in range(0, len(X_perturb), batch_size):
                        batch = X_perturb[i : (i+batch_size),samp,:,:].to('cuda')
                        for celltype, state, fold in itertools.product(models, states, folds):
                            prediction = model_dict[celltype][state][fold](batch).cpu().numpy()[:,0]
                            temp_dict[celltype][state][fold]['after'][i:(i+len(batch)), samp] = prediction
    
                # Concatenate over peakset for each model, state and fold
                for celltype, state, fold in itertools.product(models, states, folds):
                    result_dict[celltype][state][fold]['idx'] = np.concat([result_dict[celltype][state][fold]['idx'],temp_dict[celltype][state][fold]['idx']])
                    result_dict[celltype][state][fold]['before'] = np.concat([result_dict[celltype][state][fold]['before'],temp_dict[celltype][state][fold]['before']])
                    result_dict[celltype][state][fold]['after'] = np.concat([result_dict[celltype][state][fold]['after'],temp_dict[celltype][state][fold]['after'].mean(axis = 1)])
    
        # Dump the pickle file
        with open(output, "wb") as of:
            pickle.dump(result_dict, of)

In [ ]:
from scipy.stats import ttest_ind
import warnings

# Parameters
use_type = 'celltype'
counter = 0
overwrite = False
output = f"{chrombpnet_dir}/data/peaks/correlated/hits/motif_ablations.pkl"
            
# Process classes of peaks
levels = all_peaks["class"].str.split(",").explode().str.strip()
alpha = levels[levels.str.startswith("alpha_")].str.replace("^alpha", "", regex=True).unique().tolist()
beta = levels[levels.str.startswith("beta_")].str.replace("^beta", "", regex=True).unique().tolist()
levels = {"alpha": alpha, "beta": beta}
if use_type == 'PC':
    levels['alpha'] = list(set(["_" + x.split("_")[1] + "_" for x in levels['alpha']]))
    levels['beta'] = list(set(["_" + x.split("_")[1] + "_" for x in levels['beta']]))
elif use_type == 'celltype':
    levels['alpha'] = ["_"]
    levels['beta'] = ["_"]

if overwrite or not os.path.exists(output):
    with warnings.catch_warnings(), np.errstate(invalid="ignore", divide="ignore"):
        warnings.filterwarnings("ignore", message="Mean of empty slice", category=RuntimeWarning)
        for midx in tqdm(range(len(hits))):
            # Get the name of the motif
            motif_name = list(motifs.keys())[midx].split(" ")[0]
            
            # Get the motif hits
            targets = hits[midx].copy()
        
            # Read data
            with open(f"{output_path}/old/{midx}.pkl", "rb") as of:
                res = pickle.load(of)
        
            # Loop over cell types and levels
            for celltype in celltypes:
                levels_ct = levels[celltype]
                for level in levels_ct:
                    # Get peak ids for the given level and celltype
                    pattern = re.escape(f"{celltype}{level}")
                    peak_ids = all_peaks[all_peaks["class"].str.contains(pattern, regex=True)].index.tolist()
            
                    # Extract perturbation data
                    mean_dict = {}
                    for disease in ['ND', 'T2D']:
                        for fold in folds:
                            before = res[celltype][disease][fold]['before']
                            after = res[celltype][disease][fold]['after']
                            idx = res[celltype][disease][fold]['idx']
                            matches = np.where(np.isin(idx.astype(int), peak_ids))[0]
                            arr_tmp = (after[matches] - before[matches])[:,np.newaxis]
                            if fold == 0:
                                arr = arr_tmp
                            else:
                                arr = np.concatenate([arr, arr_tmp], axis = 1)
                        mean_dict[disease] = arr.mean(axis = 0)
                    
                    # Pvalue
                    x = mean_dict['T2D']
                    y = mean_dict['ND']
                    stat, p = ttest_ind(x, y)
                              
                    # Setup as pandas
                    df_tmp = pd.DataFrame({
                        "celltype": celltype,
                        "class": level,
                        "motif": motif_name,
                        "Pvalue": p,
                        "T2D_1": x[0],
                        "T2D_2": x[1],
                        "T2D_3": x[2],
                        "T2D_4": x[3],
                        "T2D_5": x[4],
                        "ND_1": y[0],
                        "ND_2": y[1],
                        "ND_3": y[2],
                        "ND_4": y[3],
                        "ND_5": y[4],
                    }, index = [counter])
            
                    # Combine
                    if counter == 0:
                        df_res = df_tmp
                    else:
                        df_res = pd.concat([df_res, df_tmp])
                    counter += 1
    
            # Dump the results as pickle file
            with open(output, "wb") as of:
                pickle.dump(df_res, of)
else:
    with open(output, "rb") as f:
        df_res = pickle.load(f)